In [104]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

fatal: destination path 'LHL-final-final-project' already exists and is not an empty directory.


In [105]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
import json
import os

In [106]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_merged_gamelogs.csv")


# preview
df.head()

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,day_of_week_by_team_travel_distance,team_vs_opp_median_score_by_team_travel_distance,team_vs_opp_homeaway_median_score_by_team_travel_distance,team_home_or_away_median_score_by_team_travel_distance,team_home_or_away_median_allowed_by_team_travel_distance,team_day_median_score_by_team_travel_distance,team_day_median_allowed_by_team_travel_distance,travel_distance_by_team_travel_distance,median_score_for_by_team_travel_distance,median_score_against_by_team_travel_distance
0,ATL,1,5,15,2,LAS,1,92,81,34,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
1,ATL,2,5,18,2,PHO,2,85,88,27,...,2.0,76.0,76.0,78.0,80.5,76.0,80.0,1.0,77.5,77.5
2,ATL,3,5,21,1,DAL,1,83,78,30,...,4.0,81.0,75.5,78.0,80.5,73.0,78.0,3.0,81.0,85.0
3,ATL,4,5,26,1,MIN,2,79,92,31,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
4,ATL,5,5,29,2,WAS,1,73,67,26,...,4.0,75.0,76.5,78.0,80.5,76.0,80.0,2.0,78.0,80.0


In [107]:
for col in df.columns:
  print(col)

team
g_num
month
day
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
offensive_four_factors_ft_per_fga
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
defensive_four_factors_ft_per_fga
day_of_week
team_vs_opp_median_score
team_vs_opp_homeaway_median_score
team_home_or_away_median_score
team_home_or_away_median_allowed
team_day_median_score
team_day_

In [108]:
# load the player game logs CSV from the data folder
player_df = pd.read_csv("LHL-final-final-project/data/player_data.csv")


# preview
player_df.head()

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,pbp_plus_minus_per_100_poss_on_off,pbp_turnovers_badpass,pbp_turnovers_lostball,pbp_fouls_committed_shoot,pbp_fouls_committed_off,pbp_fouls_drawn_shoot,pbp_fouls_drawn_off,pbp_misc_pga,pbp_misc_and1,pbp_misc_blkd
0,Diana Taurasi,2004,PHO,22.0,34.0,34.0,33.2,6.1,14.8,0.416,...,1.6,46.0,13.0,38.0,27.0,0.0,0.0,317.0,18.0,18.0
1,Diana Taurasi,2005,PHO,23.0,33.0,33.0,33.0,5.3,12.9,0.410,...,4.8,61.0,20.0,42.0,22.0,0.0,0.0,342.0,8.0,12.0
2,Diana Taurasi,2006,PHO,24.0,34.0,34.0,33.9,8.8,19.4,0.452,...,14.0,33.0,16.0,53.0,18.0,71.0,2.0,299.0,17.0,19.0
3,Diana Taurasi,2007,PHO,25.0,32.0,32.0,32.0,6.4,14.6,0.440,...,2.7,50.0,13.0,49.0,15.0,52.0,6.0,312.0,23.0,10.0
4,Diana Taurasi,2008,PHO,26.0,34.0,34.0,31.9,7.6,17.0,0.446,...,10.4,44.0,17.0,46.0,20.0,102.0,14.0,276.0,25.0,27.0


In [109]:
for col in player_df.columns:
  print(col)

player
year
tm
age
g
gs
per_game_mp
per_game_fg
per_game_fga
per_game_fg_pct
per_game_3p
per_game_3pa
per_game_3p_pct
per_game_2p
per_game_2pa
per_game_2p_pct
per_game_efg_pct
per_game_ft
per_game_fta
per_game_ft_pct
per_game_orb
per_game_drb
per_game_trb
per_game_ast
per_game_stl
per_game_blk
per_game_tov
per_game_pf
per_game_pts
mp
per_minute_fg
per_minute_fga
per_minute_fg_pct
per_minute_3p
per_minute_3pa
per_minute_3p_pct
per_minute_2p
per_minute_2pa
per_minute_2p_pct
per_minute_ft
per_minute_fta
per_minute_ft_pct
per_minute_orb
per_minute_drb
per_minute_trb
per_minute_ast
per_minute_stl
per_minute_blk
per_minute_tov
per_minute_pf
per_minute_pts
per_poss_fg
per_poss_fga
per_poss_fg_pct
per_poss_3p
per_poss_3pa
per_poss_3p_pct
per_poss_2p
per_poss_2pa
per_poss_2p_pct
per_poss_ft
per_poss_fta
per_poss_ft_pct
per_poss_orb
per_poss_drb
per_poss_trb
per_poss_ast
per_poss_stl
per_poss_blk
per_poss_tov
per_poss_pf
per_poss_pts
per_poss_ortg
per_poss_drtg
advanced_per
advanced_ts_pct
advan

In [110]:
# top 25 per_poss_pts in 2024 with selected columns
player_df[player_df["year"] == 2024] \
    .sort_values("per_poss_pts", ascending=False) \
    .loc[:, ["player", "tm", "per_game_mp", "per_poss_pts", "per_game_pts"]] \
    .head(25)

,player,tm,per_game_mp,per_poss_pts,per_game_pts
729,A'ja Wilson,LVA,34.4,39.2,26.9
762,Chennedy Carter,CHI,26.0,34.4,17.5
807,Kahleah Copper,PHO,32.4,33.4,21.1
746,Breanna Stewart,NYL,32.7,32.0,20.4
750,Brittney Griner,PHO,28.7,31.8,17.8
857,Napheesa Collier,MIN,34.7,30.3,20.4
887,Shakira Austin,WAS,19.8,30.0,11.8
815,Kelsey Mitchell,IND,32.0,30.0,19.2
800,Jewell Loyd,SEA,33.7,29.5,19.7
883,Sabrina Ionescu,NYL,32.1,29.0,18.2


In [111]:
# top 25 per_poss_pts in 2024 with selected columns
player_df[player_df["year"] == 2024] \
    .sort_values("per_game_pts", ascending=False) \
    .loc[:, ["player", "tm", "per_game_mp", "per_game_pts"]] \
    .head(25)

,player,tm,per_game_mp,per_game_pts
729,A'ja Wilson,LVA,34.4,26.9
742,Arike Ogunbowale,DAL,38.6,22.2
807,Kahleah Copper,PHO,32.4,21.1
857,Napheesa Collier,MIN,34.7,20.4
746,Breanna Stewart,NYL,32.7,20.4
800,Jewell Loyd,SEA,33.7,19.7
753,Caitlin Clark,IND,35.4,19.2
815,Kelsey Mitchell,IND,32.0,19.2
883,Sabrina Ionescu,NYL,32.1,18.2
885,Satou Sabally,DAL,34.1,17.9


In [112]:
# filter for 2024 players
df_2024 = player_df[player_df["year"] == 2024]

# full set of 27 personas and their primary sorting stat
personas = {
    # Offensive
    "elite_scorer": "per_game_pts",
    "efficient_scorer": "advanced_ts_pct",
    "volume_shooter": "per_game_fga",
    "three_point_specialist": "per_game_3pa",
    "slasher": "shooting_fg_pct_by_distance_0-3",
    "free_throw_generator": "per_game_fta",
    "and_one_machine": "pbp_misc_and1",

    # Playmaking / IQ
    "playmaker": "per_game_ast",
    "offensive_hub": "advanced_ast_pct",
    "turnover_prone": "per_game_tov",
    "floor_general": "advanced_ast_pct",  # sort by ast_pct, can display tov_pct too
    "plus_minus_driver": "pbp_plus_minus_per_100_poss_on_off",
    "self_creator": "advanced_usg_pct",  # sort by usg_pct, low %astd may be inspected separately

    # Defensive
    "rim_protector": "per_game_blk",
    "steal_artist": "per_game_stl",
    "defensive_anchor": "advanced_dws",
    "glass_cleaner": "per_game_trb",
    "offensive_rebounder": "advanced_orb_pct",
    "defensive_rebounder": "advanced_drb_pct",

    # Shooting types
    "midrange_sniper": "shooting_fg_pct_by_distance_10-16",
    "corner_3_specialist": "shooting_corner_3s_3p_pct",
    "catch_and_shoot": "shooting_pct_of_fg_astd_3p",
    "stretch_big": "per_game_3pa",
    "heave_chucker": "shooting_heaves_att",

    # Misc
    "all_around_star": "advanced_ws",
    "impact_bench": "per_minute_pts",
    "fast_break_threat": "pbp_misc_pga",
}

# collect Top 10 per persona
top_10_personas = {
    label: df_2024.sort_values(stat, ascending=False).loc[:, ["player", "tm", stat]].head(10)
    for label, stat in personas.items()
}

In [113]:
df_2024

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,pbp_plus_minus_per_100_poss_on_off,pbp_turnovers_badpass,pbp_turnovers_lostball,pbp_fouls_committed_shoot,pbp_fouls_committed_off,pbp_fouls_drawn_shoot,pbp_fouls_drawn_off,pbp_misc_pga,pbp_misc_and1,pbp_misc_blkd
729,A'ja Wilson,2024,LVA,27.0,38.0,38.0,34.4,10.1,19.6,0.518,...,-2.7,20.0,17.0,42.0,9.0,131.0,2.0,223.0,30.0,48.0
730,Aaliyah Edwards,2024,WAS,21.0,34.0,17.0,21.8,3.0,6.2,0.490,...,-7.0,18.0,13.0,42.0,15.0,33.0,10.0,122.0,4.0,23.0
731,Aari McDonald,2024,LAS,25.0,26.0,10.0,21.8,3.0,7.3,0.403,...,3.3,29.0,10.0,14.0,3.0,13.0,13.0,227.0,3.0,10.0
732,Aerial Powers,2024,ATL,30.0,17.0,2.0,17.9,2.9,8.1,0.355,...,-7.3,5.0,8.0,14.0,2.0,14.0,0.0,56.0,3.0,10.0
733,Alanna Smith,2024,MIN,27.0,39.0,39.0,26.5,3.8,8.0,0.471,...,9.7,35.0,20.0,58.0,14.0,32.0,10.0,300.0,9.0,20.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
907,Tyasha Harris,2024,CON,26.0,39.0,38.0,28.8,3.7,8.7,0.425,...,0.0,39.0,10.0,36.0,1.0,37.0,6.0,264.0,10.0,15.0
908,Veronica Burton,2024,CON,23.0,31.0,1.0,12.7,0.8,2.3,0.361,...,7.7,11.0,3.0,14.0,0.0,12.0,7.0,131.0,1.0,4.0
909,Victaria Saxton,2024,IND,24.0,9.0,0.0,2.6,0.3,1.0,0.333,...,-25.1,0.0,1.0,3.0,0.0,1.0,0.0,0.0,0.0,1.0
910,Victoria Vivians,2024,SEA,29.0,35.0,15.0,12.7,1.2,3.5,0.333,...,1.6,12.0,3.0,24.0,2.0,3.0,3.0,67.0,1.0,8.0


In [114]:
# one-hot encode persona membership: 1 if in top 10 for that persona, else 0
for persona, df_top10 in top_10_personas.items():
    top_players = set(df_top10["player"])
    player_df[persona] = player_df["player"].apply(lambda x: 1 if x in top_players else 0)

In [115]:
player_df

,player,year,tm,age,g,gs,per_game_mp,per_game_fg,per_game_fga,per_game_fg_pct,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
0,Diana Taurasi,2004,PHO,22.0,34.0,34.0,33.2,6.1,14.8,0.416,...,0,0,0,0,0,1,0,0,0,0
1,Diana Taurasi,2005,PHO,23.0,33.0,33.0,33.0,5.3,12.9,0.410,...,0,0,0,0,0,1,0,0,0,0
2,Diana Taurasi,2006,PHO,24.0,34.0,34.0,33.9,8.8,19.4,0.452,...,0,0,0,0,0,1,0,0,0,0
3,Diana Taurasi,2007,PHO,25.0,32.0,32.0,32.0,6.4,14.6,0.440,...,0,0,0,0,0,1,0,0,0,0
4,Diana Taurasi,2008,PHO,26.0,34.0,34.0,31.9,7.6,17.0,0.446,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1064,Tyasha Harris,0,Career,26.0,167.0,49.0,19.6,2.4,5.9,0.410,...,0,0,0,0,0,0,0,0,0,0
1065,Veronica Burton,0,Career,23.0,107.0,20.0,14.0,0.7,2.2,0.326,...,0,0,0,0,0,0,0,0,0,0
1066,Victaria Saxton,0,Career,24.0,24.0,0.0,3.2,0.4,1.1,0.333,...,0,0,0,0,0,0,0,0,0,0
1067,Victoria Vivians,0,Career,29.0,179.0,93.0,20.8,2.4,6.8,0.357,...,0,0,0,0,0,0,0,0,0,0


In [116]:
# load the player game logs CSV from the data folder
player_gamelogs_df = pd.read_csv("LHL-final-final-project/data/2024_player_gamelogs.csv")


# preview
player_gamelogs_df.head()

,player,year,month,day,age,tm,home_away,opp,win_margin,gs,...,orb,drb,trb,ast,stl,blk,tov,pf,pts,gmsc
0,Lindsay Allen,2024,5,15,29.2,CHI,2,DAL,-8.0,0,...,0,1,1,0,1,0,0,0,5,3.2
1,Lindsay Allen,2024,5,18,29.2,CHI,2,DAL,9.0,0,...,0,1,1,1,0,0,1,0,2,1.7
2,Lindsay Allen,2024,5,23,29.2,CHI,2,NYL,9.0,0,...,0,1,1,2,0,0,1,3,8,4.2
3,Lindsay Allen,2024,5,25,29.2,CHI,1,CON,-4.0,0,...,1,1,2,2,2,0,2,1,6,7.1
4,Lindsay Allen,2024,5,28,29.2,CHI,1,SEA,-9.0,0,...,0,2,2,4,0,0,3,2,3,0.1


In [117]:
# get list of persona columns
persona_cols = list(top_10_personas.keys())

# subset to player + persona columns
player_personas = player_df[["player"] + persona_cols]

# merge into player_gamelogs_df
player_gamelogs_df = player_gamelogs_df.merge(player_personas, on="player", how="left")

In [118]:
# filter rows for A'ja Wilson
player_gamelogs_df[player_gamelogs_df["player"] == "A'ja Wilson"]

,player,year,month,day,age,tm,home_away,opp,win_margin,gs,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
31851,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31852,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31853,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31854,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
31855,A'ja Wilson,2024,5,14,27.8,LVA,1,PHO,9.0,1,...,0,1,0,0,1,0,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32150,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32151,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32152,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0
32153,A'ja Wilson,2024,9,17,28.1,LVA,2,SEA,13.0,1,...,0,1,0,0,1,0,1,1,1,0


In [119]:
# drop duplicate player-game rows before summing personas
player_personas_clean = (
    player_gamelogs_df
    .drop_duplicates(subset=["player", "month", "day", "tm", "opp"])
)

# now group and sum by game
persona_summary = (
    player_personas_clean
    .rename(columns={"tm": "team"})
    .groupby(["month", "day", "team", "opp"])[persona_cols]
    .sum()
    .reset_index()
)

# drop existing persona columns from df before merging
df = df.drop(columns=persona_cols, errors="ignore")

# re-merge into df
df = df.merge(persona_summary, on=["month", "day", "team", "opp"], how="left")

In [120]:
df

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
0,ATL,1,5,15,2,LAS,1,92,81,34,...,1,1,0,0,0,1,2,0,0,0
1,ATL,2,5,18,2,PHO,2,85,88,27,...,1,1,0,0,0,1,2,0,0,0
2,ATL,3,5,21,1,DAL,1,83,78,30,...,0,1,0,0,0,1,2,0,0,0
3,ATL,4,5,26,1,MIN,2,79,92,31,...,1,1,0,0,0,1,2,0,0,0
4,ATL,5,5,29,2,WAS,1,73,67,26,...,0,1,0,0,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,WAS,36,9,11,2,CHI,1,89,58,34,...,1,1,2,2,1,0,1,0,0,1
476,WAS,37,9,13,2,ATL,1,72,69,26,...,1,1,1,2,1,0,1,0,0,1
477,WAS,38,9,15,1,ATL,2,73,76,23,...,1,1,1,2,1,0,0,0,0,1
478,WAS,39,9,17,1,NYL,2,71,87,23,...,1,1,2,2,1,0,0,0,0,1


In [121]:
# subset df to just the desired columns
df_personas = df[["team", "opp", "month", "day", "team_score", "opp_score"] + persona_cols]

In [122]:
for col in df_personas.columns:
  print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat


In [123]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas.copy()
    else:
        df_team = df_personas[df_personas["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,80.848381,13.848381,80,83.648148,3.648148
1,ATL,ATL,NYL,75,76.765610,1.765610,81,83.591324,2.591324
2,ATL,ATL,CON,78,75.747864,2.252136,74,95.562180,21.562180
3,ATL,ATL,PHO,72,80.851952,8.851952,63,78.111542,15.111542
4,ATL,ATL,WAS,73,81.570412,8.570412,67,83.216370,16.216370


In [124]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

In [125]:
model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.745827,28.240044,8.824952,7.447834,-1.704185,0.559769,21.562180,8.432312,4.695320,-1.863497
1,CHI,1.306015,24.273590,11.662463,11.584152,-1.689897,1.575516,22.635521,8.748167,6.590889,-0.905401
2,CON,5.687241,21.102097,14.413157,13.550289,-1.359539,0.456940,24.206276,12.110241,12.605919,-1.667487
3,DAL,4.617722,30.990234,15.720156,14.402164,-3.669330,0.179626,24.592522,12.002954,10.433495,-3.420389
4,IND,0.029259,30.707825,7.114034,4.578827,-0.926229,2.764320,22.187256,11.175971,11.010525,-6.591712
5,LAS,4.245056,25.404205,13.538713,11.033638,-1.053857,0.624245,28.042351,10.681267,6.204750,-2.237150
6,LVA,2.223869,35.554207,11.179570,7.700703,-2.227964,4.527397,21.795876,11.363002,8.739994,-0.730825
7,League,0.063721,30.678833,9.367292,8.469028,-0.096510,0.540077,22.873520,8.936762,8.251755,0.058527
8,MIN,6.150040,19.538803,12.881006,13.996567,-1.067274,2.278740,37.332977,11.816581,9.153164,-1.951541
9,NYL,0.495872,11.665970,5.349606,4.492310,0.319165,1.845360,20.852913,11.794661,13.046185,-1.222202


In [126]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

In [127]:
df_importances

,Feature,Importance_team_score,Importance_opp_score
2,elite_scorer,0.128576,0.033000
21,midrange_sniper,0.044332,0.028628
25,heave_chucker,0.043174,0.020754
20,defensive_rebounder,0.041653,0.032244
22,corner_3_specialist,0.041627,0.021160
18,glass_cleaner,0.039855,0.020463
3,efficient_scorer,0.039835,0.031098
14,self_creator,0.039708,0.021526
9,playmaker,0.037996,0.020442
0,month,0.035761,0.024182


In [128]:
# define final columns
df_personas_v2 = df[["team", "opp", "month", "day", "team_score", "opp_score", "advanced_pace"] + persona_cols].copy()

In [129]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v2.copy()
    else:
        df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,75.386131,8.386131,80,71.990845,8.009155
1,ATL,ATL,NYL,75,96.601509,21.601509,81,97.432861,16.432861
2,ATL,ATL,CON,78,74.721062,3.278938,74,86.871834,12.871834
3,ATL,ATL,PHO,72,80.815063,8.815063,63,62.977020,0.022980
4,ATL,ATL,WAS,73,83.179428,10.179428,67,79.915451,12.915451


In [130]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.595459,46.759552,12.541367,9.396435,-2.381513,0.022980,21.050987,10.005225,11.447407,-2.471699
1,CHI,0.639687,27.175789,11.568124,10.673702,-1.107820,0.000069,16.748390,5.336271,2.757290,0.059745
2,CON,2.080414,25.091003,11.079163,8.000000,-0.920188,0.689453,17.624657,7.538147,6.618820,0.218508
3,DAL,0.889320,23.445518,10.245697,10.004631,-1.790000,0.806633,22.988869,10.410671,9.642426,-1.974672
4,IND,1.249550,23.027046,9.002253,5.315155,-1.255144,0.054955,21.891342,7.453757,4.796322,-1.178824
5,LAS,0.405502,21.302246,9.925974,11.811764,-0.325172,0.502922,30.099030,8.044093,3.783710,0.086570
6,LVA,0.608078,17.423988,7.301405,6.194748,-0.295896,0.156075,25.895302,10.773399,10.794285,-0.969433
7,League,0.146263,37.380058,8.060199,6.621078,0.051046,0.083511,26.400139,8.316116,7.148979,0.037107
8,MIN,0.705803,17.725845,10.306474,10.145969,0.272668,1.544220,30.861305,10.333002,5.799826,-1.297828
9,NYL,1.948608,13.096603,6.436944,5.731804,0.370604,1.801315,22.565453,10.640791,11.248291,-1.250709


In [131]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
1,elite_scorer,0.077038,0.027455
27,fast_break_threat,0.066030,0.017198
25,all_around_star,0.063557,0.145818
3,volume_shooter,0.052194,0.047618
8,playmaker,0.050857,0.021945
21,corner_3_specialist,0.045057,0.021474
24,heave_chucker,0.042556,0.026316
26,impact_bench,0.042463,0.037559
20,midrange_sniper,0.041384,0.025218
14,rim_protector,0.037438,0.029683


In [132]:
predictions_df.query("Model != 'League'")[["team_score_mae", "opp_score_mae"]].describe()

,team_score_mae,opp_score_mae
count,120.000000,120.000000
mean,10.533257,9.090765
std,8.085162,7.224445
min,0.207840,0.000069
25%,3.512133,2.796633
50%,9.822010,6.980057
75%,16.202703,13.617031
max,46.759552,30.861305


In [133]:
# features and targets
X = df_personas_v2.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_personas_v2[["team_score", "opp_score"]]

# cross-validation setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# store fold scores
cv_results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = MultiOutputRegressor(XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    ))

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # compute MAE and R² for both targets
    for i, col in enumerate(["team_score", "opp_score"]):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])

        cv_results.append({
            "Fold": fold + 1,
            "Target": col,
            "MAE": mae,
            "R2": r2
        })

# convert to DataFrame
cv_results_df = pd.DataFrame(cv_results)

# view
cv_results_df.groupby("Target").agg({"MAE": ["mean", "std"], "R2": ["mean", "std"]})

MAE                  R2          
                mean       std      mean       std
Target                                            
opp_score   7.933787  0.746936  0.105864  0.137893
team_score  7.965906  0.864958  0.031581  0.098155

In [134]:
# list of teams
teams = df_personas_v2["team"].unique()

# store team-level CV results
team_cv_results = []

for team_name in teams:
    df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = MultiOutputRegressor(XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=1,
            min_child_weight=1,
            random_state=42,
            verbosity=0
        ))

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        for i, col in enumerate(["team_score", "opp_score"]):
            mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
            r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])

            team_cv_results.append({
                "Team": team_name,
                "Fold": fold + 1,
                "Target": col,
                "MAE": mae,
                "R2": r2
            })

# compile and summarize
team_cv_df = pd.DataFrame(team_cv_results)

# mean/std per team per target
team_cv_summary = team_cv_df.groupby(["Team", "Target"]).agg(
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    R2_mean=("R2", "mean"),
    R2_std=("R2", "std")
).reset_index()

# view
team_cv_summary.head()

,Team,Target,MAE_mean,MAE_std,R2_mean,R2_std
0,ATL,opp_score,8.073456,1.400013,-0.703656,0.741453
1,ATL,team_score,9.587510,1.873829,-1.497677,1.851252
2,CHI,opp_score,6.584826,1.406470,-0.974827,0.671103
3,CHI,team_score,10.553030,2.665447,-1.121606,1.000158
4,CON,opp_score,8.808833,0.970905,-0.380157,0.327031


In [135]:
df_team = df_personas_v2[df_personas_v2["team"] == "ATL"].copy()

# keep inputs the same
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# shuffle the targets
y_shuffled = df_team[["team_score", "opp_score"]].sample(frac=1, random_state=42).reset_index(drop=True)

In [136]:
# choose one team
team_name = "ATL"
df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

# features
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# shuffle the targets
y = df_team[["team_score", "opp_score"]].sample(frac=1, random_state=42).reset_index(drop=True)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# model
model = MultiOutputRegressor(XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# evaluate
for i, col in enumerate(["team_score", "opp_score"]):
    mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
    r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"[{team_name}] {col} — MAE: {mae:.3f}, R²: {r2:.3f}")

[ATL] team_score — MAE: 9.439, R²: -0.543
[ATL] opp_score — MAE: 12.900, R²: -1.028


In [137]:
# preview predictions vs shuffled targets
pred_df = pd.DataFrame({
    "team_score_true": y_test["team_score"].values,
    "team_score_pred": y_pred[:, 0],
    "opp_score_true": y_test["opp_score"].values,
    "opp_score_pred": y_pred[:, 1],
})

# show top rows
pred_df.head(10)

,team_score_true,team_score_pred,opp_score_true,opp_score_pred
0,107,72.921837,96,75.739265
1,77,75.815819,85,83.790878
2,89,92.153069,80,94.129890
3,72,61.586266,83,72.086525
4,73,76.155556,67,74.832397
5,79,67.358528,91,72.268028
6,75,61.586266,96,72.086525
7,69,74.789360,72,84.052277
8,86,75.950317,70,84.075142
9,76,77.511620,73,78.881973


In [138]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas.copy()
    else:
        df_team = df_personas[df_personas["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,81.886818,14.886818,80,84.766785,4.766785
1,ATL,ATL,NYL,75,77.073280,2.073280,81,85.206291,4.206291
2,ATL,ATL,CON,78,75.045906,2.954094,74,95.914780,21.914780
3,ATL,ATL,PHO,72,82.663055,10.663055,63,82.794678,19.794678
4,ATL,ATL,WAS,73,84.109818,11.109818,67,79.798813,12.798813


In [139]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348,0.794678,21.914780,10.258556,10.389462,-2.848078
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352,0.887329,16.854660,7.450861,7.538235,-0.093799
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245,0.716217,24.018150,10.653886,9.648392,-0.502846
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264,0.311890,17.824005,9.450786,9.771706,-1.708256
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784,0.382210,24.539467,8.338472,4.460838,-1.653995
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874,0.480682,38.003204,9.230894,4.272728,-0.241763
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650,1.838875,29.935532,12.023899,9.000000,-1.738078
7,League,0.018066,35.224098,8.825760,6.803436,-0.131752,0.183662,25.192322,8.628613,6.957840,0.012199
8,MIN,3.942062,19.752930,10.880116,11.007225,0.280743,0.495819,25.358147,9.141777,7.898628,-0.557453
9,NYL,1.300591,20.138535,6.471673,6.250748,0.215845,0.084229,18.152611,8.448351,6.887119,-0.547534


In [140]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_merged_gamelogs.csv")


# preview
df.head()

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,day_of_week_by_team_travel_distance,team_vs_opp_median_score_by_team_travel_distance,team_vs_opp_homeaway_median_score_by_team_travel_distance,team_home_or_away_median_score_by_team_travel_distance,team_home_or_away_median_allowed_by_team_travel_distance,team_day_median_score_by_team_travel_distance,team_day_median_allowed_by_team_travel_distance,travel_distance_by_team_travel_distance,median_score_for_by_team_travel_distance,median_score_against_by_team_travel_distance
0,ATL,1,5,15,2,LAS,1,92,81,34,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
1,ATL,2,5,18,2,PHO,2,85,88,27,...,2.0,76.0,76.0,78.0,80.5,76.0,80.0,1.0,77.5,77.5
2,ATL,3,5,21,1,DAL,1,83,78,30,...,4.0,81.0,75.5,78.0,80.5,73.0,78.0,3.0,81.0,85.0
3,ATL,4,5,26,1,MIN,2,79,92,31,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
4,ATL,5,5,29,2,WAS,1,73,67,26,...,4.0,75.0,76.5,78.0,80.5,76.0,80.0,2.0,78.0,80.0


In [141]:
team_pace_df = (
    df.groupby("team")["advanced_pace"]
    .mean()
    .rename("team_avg_advanced_pace")
    .reset_index()
)

In [142]:
team_pace_df

,team,team_avg_advanced_pace
0,ATL,77.0550
1,CHI,78.1025
2,CON,75.8175
3,DAL,80.0825
4,IND,79.7600
5,LAS,79.1750
6,LVA,79.6975
7,MIN,77.5525
8,NYL,78.0375
9,PHO,78.1200


In [143]:
# 2. Drop the game-level advanced_pace from df_personas_v2 (if it exists)
df_personas_v2 = df_personas_v2.drop(columns=["advanced_pace"], errors="ignore")

# 3. Merge in team average pace
df_personas_v2 = df_personas_v2.merge(team_pace_df, on="team", how="left")

In [144]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v2.copy()
    else:
        df_team = df_personas_v2[df_personas_v2["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,81.886818,14.886818,80,84.766785,4.766785
1,ATL,ATL,NYL,75,77.073280,2.073280,81,85.206291,4.206291
2,ATL,ATL,CON,78,75.045906,2.954094,74,95.914780,21.914780
3,ATL,ATL,PHO,72,82.663055,10.663055,63,82.794678,19.794678
4,ATL,ATL,WAS,73,84.109818,11.109818,67,79.798813,12.798813


In [145]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348,0.794678,21.914780,10.258556,10.389462,-2.848078
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352,0.887329,16.854660,7.450861,7.538235,-0.093799
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245,0.716217,24.018150,10.653886,9.648392,-0.502846
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264,0.311890,17.824005,9.450786,9.771706,-1.708256
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784,0.382210,24.539467,8.338472,4.460838,-1.653995
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874,0.480682,38.003204,9.230894,4.272728,-0.241763
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650,1.838875,29.935532,12.023899,9.000000,-1.738078
7,League,0.027824,35.329521,8.818955,6.470230,-0.114272,0.094803,26.426544,8.734950,7.082985,-0.014580
8,MIN,3.942062,19.752930,10.880116,11.007225,0.280743,0.495819,25.358147,9.141777,7.898628,-0.557453
9,NYL,1.300591,20.138535,6.471673,6.250748,0.215845,0.084229,18.152611,8.448351,6.887119,-0.547534


In [146]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.089380,0.021039
27,team_avg_advanced_pace,0.049639,0.173482
18,defensive_rebounder,0.048581,0.034738
19,midrange_sniper,0.045816,0.033638
23,heave_chucker,0.045646,0.028509
24,all_around_star,0.044239,0.061437
20,corner_3_specialist,0.043651,0.022348
9,turnover_prone,0.040815,0.030626
15,defensive_anchor,0.040632,0.108308
16,glass_cleaner,0.039248,0.031304


In [147]:
# load the player game logs CSV from the data folder
team_df_2024 = pd.read_csv("LHL-final-final-project/data/2024_team_data.csv")


# preview
team_df_2024.head()

,Team,G,MP,team_per_game_FG,team_per_game_FGA,team_per_game_FG%,team_per_game_3P,team_per_game_3PA,team_per_game_3P%,team_per_game_2P,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,Atlanta Dream,40,201.9,27.8,68.1,0.408,6.0,19.4,0.308,21.8,...,0.473,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,Chicago Sky,40,200.0,29.7,70.3,0.422,4.8,14.9,0.323,24.9,...,0.503,0.650,0.391,0.393,0.433,0.326,0.672,0.880,0.203,0.278
2,Connecticut Sun,40,201.2,29.3,65.9,0.444,5.9,18.0,0.327,23.4,...,0.488,0.660,0.435,0.401,0.362,0.313,0.644,0.895,0.193,0.346
3,Dallas Wings,40,201.9,31.7,71.0,0.446,6.3,19.2,0.326,25.4,...,0.530,0.705,0.468,0.400,0.397,0.365,0.620,0.898,0.197,0.357
4,Indiana Fever,40,200.6,31.3,68.5,0.456,9.2,25.9,0.356,22.1,...,0.487,0.608,0.448,0.375,0.408,0.361,0.603,0.889,0.193,0.396


In [148]:
# create a copy and drop the specified column
df_personas_v3 = df_personas_v2.drop(columns=["team_avg_advanced_pace"])


In [149]:
# mapping full team name to 3-letter abbreviation
team_name_to_abbr = {
    "Atlanta Dream": "ATL",
    "Chicago Sky": "CHI",
    "Connecticut Sun": "CON",
    "Dallas Wings": "DAL",
    "Indiana Fever": "IND",
    "Las Vegas Aces": "LVA",
    "Los Angeles Sparks": "LAS",
    "Minnesota Lynx": "MIN",
    "New York Liberty": "NYL",
    "Phoenix Mercury": "PHO",
    "Seattle Storm": "SEA",
    "Washington Mystics": "WAS"
}

# map and add new column
team_df_2024["team_abbr"] = team_df_2024["Team"].map(team_name_to_abbr)

In [150]:
# drop G and MP before merge
team_df_2024_filtered = team_df_2024.drop(columns=["G", "MP"])

# merge using left df 'team' and right df 'team_abbr'
persona_and_teams_df = pd.merge(df_personas_v3, team_df_2024_filtered, left_on="team", right_on="team_abbr", how="left")

In [151]:
persona_and_teams_df

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%,team_abbr
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357,ATL
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357,ATL
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357,ATL
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357,ATL
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.617,0.400,0.358,0.397,0.344,0.628,0.874,0.215,0.357,ATL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,WAS,CHI,9,11,89,58,0,1,0,0,...,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347,WAS
476,WAS,ATL,9,13,72,69,0,1,0,0,...,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347,WAS
477,WAS,ATL,9,15,73,76,0,1,0,0,...,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347,WAS
478,WAS,NYL,9,17,71,87,0,1,0,0,...,0.699,0.443,0.439,0.377,0.333,0.612,0.887,0.228,0.347,WAS


In [152]:
for col in persona_and_teams_df.columns:
    print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
Team
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
team_totals_2P%
team_totals_FT
team_t

In [153]:
# drop team_abbr
persona_and_teams_df = persona_and_teams_df.drop(columns=["team_abbr"])

In [154]:
# print all columns with at least 1 null value
for col in persona_and_teams_df.columns:
    if persona_and_teams_df[col].isnull().any():
        print(col)
else:
    print("Done checking nulls.")

Done checking nulls.


In [155]:
# drop team_abbr
persona_and_teams_df = persona_and_teams_df.drop(columns=["Team"])

In [156]:
for col in persona_and_teams_df.columns:
    print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
team_totals_2P%
team_totals_FT
team_totals

In [157]:
persona_and_teams_df.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357


In [158]:
df_personas_v3

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,offensive_rebounder,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat
0,ATL,LAS,5,15,92,81,0,0,1,1,...,1,1,0,0,0,1,2,0,0,0
1,ATL,PHO,5,18,85,88,0,0,1,1,...,1,1,0,0,0,1,2,0,0,0
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0,1,0,0,0,1,2,0,0,0
3,ATL,MIN,5,26,79,92,0,0,1,1,...,1,1,0,0,0,1,2,0,0,0
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0,1,0,0,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,WAS,CHI,9,11,89,58,0,1,0,0,...,1,1,2,2,1,0,1,0,0,1
476,WAS,ATL,9,13,72,69,0,1,0,0,...,1,1,1,2,1,0,1,0,0,1
477,WAS,ATL,9,15,73,76,0,1,0,0,...,1,1,1,2,1,0,0,0,0,1
478,WAS,NYL,9,17,71,87,0,1,0,0,...,1,1,2,2,1,0,0,0,0,1


In [159]:
# save to CSV
persona_and_teams_df.to_csv("2024_persona_and_team_data.csv", index=False)

# download to local machine
from google.colab import files
files.download("2024_persona_and_team_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [179]:
df_personas_v3 = df_personas_v2.merge(
    persona_and_teams_df[['team', 'month', 'day', 'team_Pace']],
    on=['team', 'month', 'day'],
    how='left'
)

In [187]:
# create a copy and drop the specified column
df_personas_v3 = df_personas_v3.drop(columns=["team_avg_advanced_pace"])

In [188]:
df_personas_v3.shape

(480, 34)

In [189]:
df_personas_v3.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,defensive_rebounder,midrange_sniper,corner_3_specialist,catch_and_shoot,stretch_big,heave_chucker,all_around_star,impact_bench,fast_break_threat,team_Pace
0,ATL,LAS,5,15,92,81,0,0,1,1,...,1,0,0,0,1,2,0,0,0,77.1
1,ATL,PHO,5,18,85,88,0,0,1,1,...,1,0,0,0,1,2,0,0,0,77.1
2,ATL,DAL,5,21,83,78,0,0,1,1,...,1,0,0,0,1,2,0,0,0,77.1
3,ATL,MIN,5,26,79,92,0,0,1,1,...,1,0,0,0,1,2,0,0,0,77.1
4,ATL,WAS,5,29,73,67,0,0,1,1,...,1,0,0,0,1,2,0,0,0,77.1


In [190]:
for col in df_personas_v3.columns:
    print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_Pace


In [191]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v3["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v3.copy()
    else:
        df_team = df_personas_v3[df_personas_v3["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,81.886818,14.886818,80,84.766785,4.766785
1,ATL,ATL,NYL,75,77.073280,2.073280,81,85.206291,4.206291
2,ATL,ATL,CON,78,75.045906,2.954094,74,95.914780,21.914780
3,ATL,ATL,PHO,72,82.663055,10.663055,63,82.794678,19.794678
4,ATL,ATL,WAS,73,84.109818,11.109818,67,79.798813,12.798813


In [192]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348,0.794678,21.914780,10.258556,10.389462,-2.848078
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352,0.887329,16.854660,7.450861,7.538235,-0.093799
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245,0.716217,24.018150,10.653886,9.648392,-0.502846
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264,0.311890,17.824005,9.450786,9.771706,-1.708256
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784,0.382210,24.539467,8.338472,4.460838,-1.653995
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874,0.480682,38.003204,9.230894,4.272728,-0.241763
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650,1.838875,29.935532,12.023899,9.000000,-1.738078
7,League,0.026077,34.805916,8.802867,6.513824,-0.113548,0.083656,26.873573,8.682082,7.069447,-0.008541
8,MIN,3.942062,19.752930,10.880116,11.007225,0.280743,0.495819,25.358147,9.141777,7.898628,-0.557453
9,NYL,1.300591,20.138535,6.471673,6.250748,0.215845,0.084229,18.152611,8.448351,6.887119,-0.547534


In [193]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.087291,0.018493
27,team_Pace,0.050916,0.152706
18,defensive_rebounder,0.048559,0.030986
19,midrange_sniper,0.046052,0.032666
24,all_around_star,0.045917,0.119592
23,heave_chucker,0.045907,0.025744
20,corner_3_specialist,0.044108,0.021124
9,turnover_prone,0.043817,0.028638
15,defensive_anchor,0.040488,0.117254
16,glass_cleaner,0.039104,0.026460


In [194]:
persona_and_teams_df.shape

(480, 218)

In [195]:
persona_and_teams_df.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357


In [196]:
# create a copy and drop the specified column
df_personas_v3 = df_personas_v3.drop(columns=["floor_general"])

In [197]:
# create a copy and drop the specified column
df_personas_v3 = df_personas_v3.drop(columns=["stretch_big"])

In [198]:
# safe merge with game-level context
df_personas_v4 = df_personas_v3.merge(
    persona_and_teams_df[['team', 'month', 'day', 'team_ORtg']],
    on=['team', 'month', 'day'],
    how='left'
)

In [199]:
df_personas_v4.shape

(480, 33)

In [200]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v4["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v4.copy()
    else:
        df_team = df_personas_v4[df_personas_v4["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,81.886818,14.886818,80,84.766785,4.766785
1,ATL,ATL,NYL,75,77.073280,2.073280,81,85.206291,4.206291
2,ATL,ATL,CON,78,75.045906,2.954094,74,95.914780,21.914780
3,ATL,ATL,PHO,72,82.663055,10.663055,63,82.794678,19.794678
4,ATL,ATL,WAS,73,84.109818,11.109818,67,79.798813,12.798813


In [201]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348,0.794678,21.914780,10.258556,10.389462,-2.848078
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352,0.887329,16.854660,7.450861,7.538235,-0.093799
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245,0.716217,24.018150,10.653886,9.648392,-0.502846
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264,0.311890,17.824005,9.450786,9.771706,-1.708256
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784,0.382210,24.539467,8.338472,4.460838,-1.653995
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874,0.480682,38.003204,9.230894,4.272728,-0.241763
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650,1.838875,29.935532,12.023899,9.000000,-1.738078
7,League,0.170532,35.458786,8.768110,6.457333,-0.100119,0.067871,26.174438,8.694020,7.050549,-0.008197
8,MIN,3.942062,19.752930,10.880116,11.007225,0.280743,0.495819,25.358147,9.141777,7.898628,-0.557453
9,NYL,1.300591,20.138535,6.471673,6.250748,0.215845,0.084229,18.152611,8.448351,6.887119,-0.547534


In [202]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.093308,0.018694
26,team_ORtg,0.049853,0.027407
25,team_Pace,0.049644,0.148359
17,defensive_rebounder,0.046066,0.031828
19,corner_3_specialist,0.044821,0.019447
18,midrange_sniper,0.043907,0.030560
21,heave_chucker,0.041907,0.025523
9,turnover_prone,0.040747,0.022214
22,all_around_star,0.037138,0.137113
23,impact_bench,0.036653,0.025679


In [203]:
# safe merge with game-level context
df_personas_v5 = df_personas_v4.merge(
    persona_and_teams_df[['team', 'month', 'day', 'team_DRtg']],
    on=['team', 'month', 'day'],
    how='left'
)

In [204]:
df_personas_v5

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,midrange_sniper,corner_3_specialist,catch_and_shoot,heave_chucker,all_around_star,impact_bench,fast_break_threat,team_Pace,team_ORtg,team_DRtg
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0,0,0,2,0,0,0,77.1,99.0,102.5
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0,0,0,2,0,0,0,77.1,99.0,102.5
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0,0,0,2,0,0,0,77.1,99.0,102.5
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0,0,0,2,0,0,0,77.1,99.0,102.5
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0,0,0,2,0,0,0,77.1,99.0,102.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,WAS,CHI,9,11,89,58,0,1,0,0,...,2,2,1,1,0,0,1,79.1,99.7,103.4
476,WAS,ATL,9,13,72,69,0,1,0,0,...,1,2,1,1,0,0,1,79.1,99.7,103.4
477,WAS,ATL,9,15,73,76,0,1,0,0,...,1,2,1,0,0,0,1,79.1,99.7,103.4
478,WAS,NYL,9,17,71,87,0,1,0,0,...,2,2,1,0,0,0,1,79.1,99.7,103.4


In [205]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = df_personas_v5["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_personas_v5.copy()
    else:
        df_team = df_personas_v5[df_personas_v5["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        tree_method="gpu_hist",
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [03:26:18] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [03:26:19] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [03:26:20] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserW

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,82.052437,15.052437,80,84.834465,4.834465
1,ATL,ATL,NYL,75,76.963760,1.963760,81,85.138939,4.138939
2,ATL,ATL,CON,78,74.763947,3.236053,74,96.118958,22.118958
3,ATL,ATL,PHO,72,83.154045,11.154045,63,83.378395,20.378395
4,ATL,ATL,WAS,73,84.380890,11.380890,67,79.615402,12.615402


In [206]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,1.074280,27.966438,10.884869,9.154045,-1.196932,1.378395,22.118958,10.358402,10.125252,-2.900999
1,CHI,1.549026,25.846893,11.616725,12.905724,-0.911378,0.789833,16.729660,7.473885,7.587994,-0.093338
2,CON,0.439842,25.624855,11.147630,11.617237,-1.258388,0.794449,23.877678,10.621177,9.594055,-0.487672
3,DAL,2.641190,38.979584,9.749972,5.502106,-2.540946,0.144173,18.382988,9.701848,10.505215,-1.840423
4,IND,1.083252,18.010643,7.547368,6.083252,-0.278256,0.279358,24.368317,8.170502,4.323837,-1.628989
5,LAS,0.045815,20.968369,9.027074,8.007431,-0.174001,0.019913,38.318382,9.379048,4.555496,-0.269663
6,LVA,0.032547,21.875664,8.277674,6.374840,-0.686226,2.062332,29.109093,11.742184,9.000000,-1.598385
7,League,0.118141,34.602448,8.764486,6.332172,-0.098475,0.031036,26.455284,8.737208,7.171486,-0.022760
8,MIN,3.307877,19.568275,10.800317,11.125973,0.292024,0.593124,25.115547,8.896388,7.692856,-0.510000
9,NYL,1.549438,20.323143,6.080458,4.435162,0.246824,0.537369,18.074272,8.034181,6.915806,-0.428591


In [207]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
0,elite_scorer,0.086799,0.013978
22,all_around_star,0.056677,0.016522
26,team_ORtg,0.054750,0.027821
25,team_Pace,0.045839,0.089133
21,heave_chucker,0.045708,0.026125
19,corner_3_specialist,0.041761,0.018028
14,defensive_anchor,0.040698,0.029901
18,midrange_sniper,0.040665,0.026400
9,turnover_prone,0.039763,0.024104
17,defensive_rebounder,0.036443,0.032560


In [208]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = persona_and_teams_df["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = persona_and_teams_df.copy()
    else:
        df_team = persona_and_teams_df[persona_and_teams_df["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,80.387367,13.387367,80,84.933357,4.933357
1,ATL,ATL,NYL,75,75.890015,0.890015,81,79.242607,1.757393
2,ATL,ATL,CON,78,75.745178,2.254822,74,94.034821,20.034821
3,ATL,ATL,PHO,72,81.310318,9.310318,63,78.326820,15.326820
4,ATL,ATL,WAS,73,81.580009,8.580009,67,81.798851,14.798851


In [209]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.757225,29.812035,8.424238,6.772343,-0.455464,0.585915,20.034821,7.416322,5.195290,-1.386258
1,CHI,1.280800,23.603577,11.663660,11.791416,-0.886500,2.515907,24.515259,9.043398,8.123634,-0.716388
2,CON,4.917160,22.346535,12.066920,10.954891,-0.955727,0.329514,25.962189,12.567095,11.420597,-1.247764
3,DAL,2.515198,29.219170,13.314982,12.758244,-3.142551,0.146904,23.792397,10.899113,7.831661,-2.833940
4,IND,0.053421,29.498100,9.576354,9.197346,-1.522313,2.811470,23.799553,11.147948,8.533688,-2.811325
5,LAS,3.674629,23.204971,13.383099,12.020592,-1.032452,0.538368,32.083405,11.038409,8.738110,-0.264706
6,LVA,1.187874,32.180298,10.011755,8.261703,-1.371256,5.051300,24.196602,12.186251,13.492493,-1.237078
7,League,0.030525,33.485382,9.142122,8.630062,-0.104613,0.060303,24.835258,8.879651,8.487335,-0.008005
8,MIN,3.169075,28.168427,12.741802,12.453129,-0.078451,4.542969,37.464592,10.715562,7.871967,-1.414342
9,NYL,0.520821,14.933357,6.011267,4.488647,0.369220,2.991364,20.170113,10.749927,10.410942,-1.317265


In [210]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
49,team_per_game_PTS,0.035484,0.000000
2,elite_scorer,0.025406,0.004484
29,team_per_game_FG,0.021936,0.011349
37,team_per_game_2P%,0.020925,0.005841
149,opp_per_game_STL,0.018053,0.007460
...,...,...,...
192,opp_per_poss_BLK,0.000000,0.000000
190,opp_per_poss_AST,0.000000,0.000000
203,opp_shooting_% of FGA by Distance_3P,0.000000,0.000000
204,opp_shooting_FG% by Distance_2P,0.000000,0.000000


In [212]:
# drop columns from persona_and_teams_df with importance < 0.01 in both targets
low_importance_cols = df_importances[
    (df_importances["Importance_team_score"] < 0.01) &
    (df_importances["Importance_opp_score"] < 0.01)
]["Feature"].tolist()

filtered_persona_and_teams_df = persona_and_teams_df.drop(columns=low_importance_cols, errors="ignore")

In [213]:
# targets
target_cols = ["team_score", "opp_score"]

# models to train: 12 teams + league-wide
teams = filtered_persona_and_teams_df["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_persona_and_teams_df.copy()
    else:
        df_team = filtered_persona_and_teams_df[filtered_persona_and_teams_df["team"] == team_name].copy()

    # features: drop non-feature columns
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score"])
    y = df_team[["team_score", "opp_score"]]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # track team + opp info from original df
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score", "opp_score"]].reset_index(drop=True)

    # model
    base_model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # predict
    y_pred = model.predict(X_test)

    # compute row-level absolute errors
    team_score_mae = np.abs(y_pred[:, 0] - meta["team_score"].values)
    opp_score_mae = np.abs(y_pred[:, 1] - meta["opp_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i, 0],
            "team_score_mae": team_score_mae[i],
            "opp_score": meta.loc[i, "opp_score"],
            "opp_score_pred": y_pred[i, 1],
            "opp_score_mae": opp_score_mae[i],
        })

    # overall summary metrics
    for i, col in enumerate(target_cols):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        mse = mean_squared_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mse)
        results.append({
            "Model": team_name,
            "Target": col,
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse
        })

# create final prediction DataFrame
predictions_df = pd.DataFrame(all_predictions)

# preview
predictions_df.head()

,Model,team,opp,team_score,team_score_pred,team_score_mae,opp_score,opp_score_pred,opp_score_mae
0,ATL,ATL,CON,67,79.125839,12.125839,80,82.089264,2.089264
1,ATL,ATL,NYL,75,79.125839,4.125839,81,82.089264,1.089264
2,ATL,ATL,CON,78,79.125839,1.125839,74,82.089264,8.089264
3,ATL,ATL,PHO,72,79.125839,7.125839,63,82.089264,19.089264
4,ATL,ATL,WAS,73,79.125839,6.125839,67,82.089264,15.089264


In [214]:
rows = []

for team, group in predictions_df.groupby('Model'):
    y_team_true = group['team_score']
    y_team_pred = group['team_score_pred']
    y_opp_true = group['opp_score']
    y_opp_pred = group['opp_score_pred']

    rows.append({
        'Model': team,
        'team_score_mae_min': (y_team_true - y_team_pred).abs().min(),
        'team_score_mae_max': (y_team_true - y_team_pred).abs().max(),
        'team_score_mae_mean': (y_team_true - y_team_pred).abs().mean(),
        'team_score_mae_median': (y_team_true - y_team_pred).abs().median(),
        'team_score_r2': r2_score(y_team_true, y_team_pred),
        'opp_score_mae_min': (y_opp_true - y_opp_pred).abs().min(),
        'opp_score_mae_max': (y_opp_true - y_opp_pred).abs().max(),
        'opp_score_mae_mean': (y_opp_true - y_opp_pred).abs().mean(),
        'opp_score_mae_median': (y_opp_true - y_opp_pred).abs().median(),
        'opp_score_r2': r2_score(y_opp_true, y_opp_pred),
    })

model_mae_r2_df = pd.DataFrame(rows)

model_mae_r2_df

,Model,team_score_mae_min,team_score_mae_max,team_score_mae_mean,team_score_mae_median,team_score_r2,opp_score_mae_min,opp_score_mae_max,opp_score_mae_mean,opp_score_mae_median,opp_score_r2
0,ATL,0.874161,29.125839,8.900671,5.125839,-0.814857,0.089264,19.089264,9.689264,11.089264,-2.321509
1,CHI,0.930618,25.069382,9.913876,9.000000,-0.398478,0.419571,12.580429,7.116086,7.000000,-0.020479
2,CON,1.990623,39.915024,11.439304,6.509377,-1.823733,0.079964,29.982384,9.860162,7.920036,-0.577590
3,DAL,2.433479,20.608177,9.212497,8.087349,-1.156830,0.113029,20.020561,7.717471,5.953766,-0.977110
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327,0.320244,24.591423,9.242949,8.500000,-1.778540
5,LAS,1.486717,19.486717,8.700000,8.000000,-0.000002,0.653122,35.346878,9.300000,6.000000,-0.069281
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116,0.551773,16.448227,7.959970,7.248077,-0.116678
7,League,0.030731,37.460426,8.548834,7.481224,-0.019625,0.116737,28.482643,8.415516,7.009304,0.040610
8,MIN,1.260544,24.861389,11.107976,9.500000,0.169617,1.527504,19.527504,8.202412,6.527504,-0.164428
9,NYL,5.981819,19.981819,9.528625,7.981819,-0.240950,2.299522,14.299522,8.088475,8.700478,-0.262236


In [215]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract and format for team_score
importances_team = model.estimators_[0].feature_importances_
importances_opp = model.estimators_[1].feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team,
    "Importance_opp_score": importances_opp
})

# sort if desired
df_importances = df_importances.sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score,Importance_opp_score
11,team_per_game_PTS,0.074638,0.002244
8,team_per_game_BLK,0.039109,0.008695
34,opp_per_game_2PA,0.038117,0.008291
22,team_DRB%,0.038060,0.008810
33,opp_per_game_2P,0.037933,0.007683
12,team_totals_AST,0.036977,0.007944
4,team_per_game_2P%,0.035913,0.004206
6,team_per_game_DRB,0.035782,0.004766
28,team_shooting_% of FGA by Distance_3-10,0.034614,0.007572
37,opp_per_poss_2PA,0.031373,0.007858
